# Ettin-Encoder-1b Lora FineTune

## Created by <a href="https://github.com/yunsuxiaozi">yunsuxiaozi</a> 2025/10/29


This notebook is a follow-up of my  notebook <a href="https://www.kaggle.com/code/yunsuxiaozi/ettin-encoder-400m-nlpstarter-lb-0-826">Ettin-Encoder-400m NLPStarter(LB:0.826)</a> and for Kaggle's <a href="https://www.kaggle.com/competitions/nlp-getting-started">NLP Beginner Competition</a>.

Due to the fact that training the model with all parameters would exceed the cuda's memory, Lora is used here to finetune the model.


## Download Model



In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

snapshot_download(
    repo_id="jhu-clsp/ettin-encoder-1b",
    local_dir="./ettin-encoder-1b",
    local_dir_use_symlinks=False,
    resume_download=True
)
print("Model download completed!")

## Config

Due to the simplicity of this dataset, training for one epoch can achieve a score of 0.838, and training for two epochs can achieve a score of 0.844.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
VER=1
model_name = "/kaggle/working/ettin-encoder-1b"
EPOCHS = 2
DIR = f"ver_{VER}"
os.makedirs(DIR, exist_ok=True)

## Load Train data

In [ ]:
import pandas as pd, numpy as np
train=pd.read_csv("/kaggle/input/nlp-getting-started/train.csv").rename(columns={'target':'label'})
print(train.shape)
train.head()

## Train with Transformers¶
We will train our Ettin model using Transformers library. The text here is relatively short, setting max_1en to 64 is sufficient.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer,AutoTokenizer,DataCollatorWithPadding
from sklearn.model_selection import train_test_split
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(model_name)
MAX_LEN = 64

## Create 20% Validation Subset

fit with full data can get higher score.

In [ ]:
# Split into train and validation sets
train_df, val_df = train_test_split(train, test_size=0.2, random_state=42)

COLS = ['text','label']
train_ds = Dataset.from_pandas(train_df[COLS])
val_ds = Dataset.from_pandas(val_df[COLS])

#fit with full data can get higher score.
# train_ds = Dataset.from_pandas(train[COLS])
# val_ds = Dataset.from_pandas(train[COLS])

## Tokenize Train Data

In [ ]:
# Tokenization function
def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=MAX_LEN)

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

columns = ['input_ids', 'attention_mask', 'label']
train_ds.set_format(type='torch', columns=columns)
val_ds.set_format(type='torch', columns=columns)

## Initialize and Train Model
Let's initialize and train our model with HuggingFace trainer. We also define a custom metric of F1 which is the competition metric.

In [ ]:
from transformers import AutoConfig, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,     
    reference_compile=False
)
# The model's config might not have pad_token_id, set it from the tokenizer
model.config.pad_token_id = tokenizer.pad_token_id

In addition to setting the parameters for Lora fine-tuning, several other important layers of the model have also been thawed here.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# Define the LoRA configuration
r=64
lora_config = LoraConfig(
    r=r,  # The rank of the LoRA matrices. A higher rank means more trainable parameters.
    lora_alpha=r, # A scaling factor for the LoRA matrices.
    lora_dropout=0.01, # Dropout probability for LoRA layers.
    bias="none",
    task_type=TaskType.SEQ_CLS, # Specify the task type for sequence classification
    target_modules=[
        "query",        # attention Q
        "key",          # attention K
        "value",        # attention V
        "dense",        # attention output dense
        'intermediate.dense','output.dense'
    ]
)

# Wrap the base model with the PEFT model
model = get_peft_model(model, lora_config)

In [ ]:
def unfreeze_critical_layers(model):
    
    layers_to_unfreeze = []
    
    for name, param in model.named_parameters():
        if any(x in name.lower() for x in ['norm', 'layer_norm', 'ln']):
            if not param.requires_grad:
                param.requires_grad = True
                layers_to_unfreeze.append(name)
    
    for name, param in model.named_parameters():
        if any(x in name for x in ['layers.21.', 'layers.22.']):
            if not param.requires_grad:
                param.requires_grad = True
                layers_to_unfreeze.append(name)
    
    for name, param in model.named_parameters():
        if any(x in name.lower() for x in ['classifier', 'score', 'head']):
            if not param.requires_grad:
                param.requires_grad = True
                layers_to_unfreeze.append(name)
    
    return layers_to_unfreeze

unfrozen_layers = unfreeze_critical_layers(model)

for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

model.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir = f"./{DIR}",
    do_train=True,
    do_eval=True,
    eval_strategy="steps",
    save_strategy="steps", #no for no saving 
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    logging_dir="./logs",
    logging_steps=50,
    save_steps=50,
    eval_steps=50,
    save_total_limit=1,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    load_best_model_at_end=True,
    report_to="none",
    gradient_accumulation_steps=4,
)

In [ ]:
from sklearn.metrics import f1_score

data_collator = DataCollatorWithPadding(tokenizer, return_tensors="pt")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, preds, average="macro")
    return {"eval_f1": f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,          
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


trainer.train()

## Create Submission CSV¶

We apply data preprocessing from the training set to the testing set to complete the prediction.

In [ ]:
test = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')
ds_test = Dataset.from_pandas(test[['text']])
ds_test = ds_test.map(tokenize, batched=True)

predictions = trainer.predict(ds_test)
probs = torch.tensor(predictions.predictions).numpy()
preds=np.argmax(probs,axis=1)
sub=pd.read_csv("/kaggle/input/nlp-getting-started/sample_submission.csv")
sub['target']=preds
sub.to_csv("ettin_encoder_1b.csv",index=None)
sub.head()